# 11교시 | RunPod 환경 전환 & Qwen3-VL 준비
**환경**: RunPod JupyterLab (GPU: NVIDIA A40) | **방법**: 실습 + 이론

---

## 학습 목표

- RunPod JupyterLab URL로 접속하여 환경 전환을 완료한다.
- Qwen3-VL 모델 로딩 및 패키지 설치를 완료한다.
- Qwen3-VL의 아키텍처 특징을 기존 VLM(Vision Language Model, LLaVA, BLIP-2, InternVL)과 비교하여 설명한다.

### 처음 읽는 분을 위한 안내

- 이번 교시를 한 줄로 보면: 대형 멀티모달 모델을 다루기 위한 환경 전환과 핵심 구조 맛보기를 하는 시간이다.
- 처음에는 여기까지 이해하면 충분하다: 왜 RunPod A40이 필요한지, Qwen3-VL이 이미지와 텍스트를 함께 처리하는 VLM(Vision Language Model)이라는 점만 잡아도 된다.
- 헷갈려도 괜찮은 부분: M-RoPE(Multimodal Rotary Position Embedding), DeepStack 같은 새 용어가 바로 익숙하지 않아도 된다. 먼저 "더 큰 모델이라 더 강한 환경이 필요하다"는 맥락을 이해하는 것이 우선이다.

### 용어 미니사전

| 용어 | 아주 쉽게 말하면 |
|---|---|
| RunPod | 큰 GPU를 원격으로 쓰는 실습 환경 |
| JupyterLab | Notebook과 Terminal을 함께 쓰는 작업 화면 |
| A40 | 48GB VRAM(Video RAM)을 가진 GPU(Graphics Processing Unit) 장비 |
| M-RoPE(Multimodal Rotary Position Embedding) | 이미지 패치의 2차원 위치를 더 잘 표현하는 방식 |
| DeepStack | 전체 보기와 확대 보기를 함께 쓰는 고해상도 처리 전략 |
| Thinking 모드 | 답만 내기 전에 추론 과정을 더 길게 드러내는 모드 |

---

## RunPod 접속

### 자주 막히는 오류

- JupyterLab이 열려도 GPU(Graphics Processing Unit)가 안 잡힘: 제공받은 인스턴스가 맞는지와 연결된 세션이 맞는지 먼저 확인한다.
- `flash-attn` 설치 실패: CUDA 환경이나 설치 순서 문제일 수 있어 오류 메시지를 먼저 확인한다.
- 모델 경로 오류: `/workspace/AE.2.1/models/...` 경로가 실제로 존재하는지 파일 탐색기와 Terminal에서 함께 확인한다.
- VRAM(Video RAM)이 예상보다 높음: 이전 세션에서 남은 모델이나 텐서가 있는지 먼저 확인한다.

### 실패했을 때 체크 순서

1. 환경 문제인지 모델 문제인지 먼저 구분한다: 접속, GPU(Graphics Processing Unit), 설치, 경로는 해결 지점이 서로 다르다.
2. Terminal과 Notebook에서 같은 정보를 교차 확인한다: GPU(Graphics Processing Unit) 이름, 경로, 설치 상태를 한쪽만 보고 넘기지 않는다.
3. 가장 앞 단계부터 다시 확인한다: JupyterLab 접속 → `uv`/`.venv` 확인 → 커널 선택 확인 → GPU(Graphics Processing Unit) 확인 → 패키지 설치 → 모델 로딩 순서로 복구한다.

이번 교시는 이전까지 Colab에서 하던 실습을 **RunPod A40 환경으로 옮기는 전환 구간**이다.  
교육생은 직접 Pod를 생성하는 대신, **RunPod의 NVIDIA A40 서버**와 **개별 JupyterLab 접속 URL**을 제공받는다.  
즉, 단순히 새 플랫폼에 로그인하는 시간이 아니라, 이후 Qwen3-VL-8B 실습이 안정적으로 돌아가도록 **GPU, 패키지, 모델 경로, JupyterLab 사용 방식**을 한 번에 정리하는 단계라고 이해하면 된다.

### Step 0. RunPod 접속 순서

```
1. 교육생별로 제공된 JupyterLab URL 확인
2. 브라우저에서 해당 URL 접속
3. JupyterLab 화면이 바로 열리는지 확인
4. Launcher 또는 파일 탐색기가 보이는지 확인
```

예시 URL

```text
https://nsowbrytz38sjn-8888.proxy.runpod.net
```

이 단계에서 확인할 점:

- 제공받은 개별 URL로 JupyterLab까지 정상적으로 진입했는가?
- 이후 실습이 Colab이 아니라 RunPod JupyterLab 기준으로 진행된다는 점을 인지했는가?

### Step 1. JupyterLab 화면 구성 이해

```
┌─────────────────────────────────────────────────┐
│  JupyterLab                                      │
│  ┌──────────┐  ┌──────────────────────────────┐ │
│  │ 파일탐색 │  │  Launcher                    │ │
│  │          │  │  ┌──────────┐ ┌──────────┐  │ │
│  │/workspace│  │  │ Notebook │ │ Terminal │  │ │
│  │  models/ │  │  │ (실습용) │ │ (설치용) │  │ │
│  │          │  │  └──────────┘ └──────────┘  │ │
│  └──────────┘  └──────────────────────────────┘ │
└─────────────────────────────────────────────────┘
```

파일 탐색기, Notebook, Terminal의 역할을 구분해 두는 것이 중요하다.  
이번 과정에서는 모든 작업을 `/workspace/AE.2.1` 아래에서 진행한다. **Terminal은 환경 준비와 패키지 설치**, **Notebook은 모델 로딩과 추론 실험** 용도로 분리해 쓰면 가장 안정적이다.

### Step 2. 작업 폴더와 `uv` 가상환경 준비 — Terminal 탭에서 실행

```bash
# 작업 폴더로 이동
cd /workspace/AE.2.1

# uv 설치 확인 (없으면 설치)
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh

# 셸에 uv 경로 반영
export PATH="$HOME/.local/bin:$PATH"

# 가상환경 생성
uv venv .venv

# 가상환경 활성화
source .venv/bin/activate

# 모델 저장 폴더 준비
mkdir -p /workspace/AE.2.1/models

# Notebook 커널 등록용 패키지 설치
uv pip install ipykernel

# JupyterLab에서 선택할 커널 등록
python -m ipykernel install --user --name ae21-qwen3vl --display-name "Python (.venv) AE.2.1"
```

이 단계에서 확인할 점:

- 모든 작업 경로가 `/workspace/AE.2.1`로 맞춰졌는가?
- `.venv`가 생성되고 활성화되었는가?
- JupyterLab에서 선택할 `Python (.venv) AE.2.1` 커널이 등록되었는가?

### Step 3. 환경 확인 — Terminal 탭에서 실행

```bash
# 작업 폴더와 가상환경 재확인
cd /workspace/AE.2.1
source .venv/bin/activate

# GPU 상태 확인
nvidia-smi

# Python 버전 확인
python --version

# uv 버전 확인
uv --version

# 실습 환경 기본 동작 확인
python -c "import sys; print(sys.executable)"

# 모델 폴더 존재 여부 확인
ls -lh /workspace/AE.2.1/models/

# 모델 다운로드 완료 여부 확인
if [[ -d /workspace/AE.2.1/models/Qwen3-VL-8B-Instruct ]]; then
  ls -lh /workspace/AE.2.1/models/Qwen3-VL-8B-Instruct/
  du -sh /workspace/AE.2.1/models/Qwen3-VL-8B-Instruct/
else
  echo "Qwen3-VL-8B-Instruct 모델이 아직 /workspace/AE.2.1/models 아래에 없습니다."
fi
```

**기대 출력 예시 1: 모델이 이미 준비된 경우**
```
+-------+----------------------+----------------------+
| GPU 0 | NVIDIA A40           | MiB / 49140 MiB      |
+-------+----------------------+----------------------+

/workspace/AE.2.1/.venv/bin/python

drwxr-xr-x 2 root root 4096 May 12 10:00 Qwen3-VL-8B-Instruct/

-rw-r--r-- config.json
-rw-r--r-- model-00001-of-00008.safetensors
...
16G  /workspace/AE.2.1/models/Qwen3-VL-8B-Instruct/
```

**기대 출력 예시 2: 모델이 아직 준비되지 않은 경우**

```
total 0
Qwen3-VL-8B-Instruct 모델이 아직 /workspace/AE.2.1/models 아래에 없습니다.
```

이 단계에서 확인할 점:

- A40 GPU가 실제로 연결되어 있는가?
- `uv`와 `.venv` 기반 실습 환경이 정상적으로 동작하는가?
- Python 실행 경로와 모델 경로가 정상적인가?
- 모델 파일이 미리 다운로드되어 있는가? 없다면 이후 로딩 전에 먼저 받아야 한다.

---

## 패키지 설치 — Terminal

### Step 4. 필수 패키지 설치

```bash
# 작업 폴더와 가상환경 재확인
cd /workspace/AE.2.1
source .venv/bin/activate

# flash-attn 빌드용 기본 도구 설치
uv pip install setuptools wheel

# PyTorch / Torchvision 설치
uv pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128

# 필수 패키지 설치
uv pip install "transformers>=4.57.0" \
               accelerate \
               matplotlib \
               qwen-vl-utils==0.0.14

# flash-attn 설치
uv pip install flash-attn --no-build-isolation -q

# 설치 확인
python -c "
import accelerate, matplotlib, transformers, torch
import torchvision

print(f'accelerate: {accelerate.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'torch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'torch CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
"
```

이 단계에서 확인할 점:

- `setuptools`, `wheel`이 `.venv`에 먼저 설치되었는가?
- `torch==2.8.0`이 `.venv`에 먼저 설치되었는가?
- `torchvision==0.23.0`이 `.venv`에 함께 설치되었는가?
- `accelerate`가 `.venv`에 설치되었는가?
- `matplotlib`가 `.venv`에 설치되었는가?
- `uv pip install`로 `transformers`, `accelerate`, `matplotlib`, `qwen-vl-utils`, `flash-attn`가 `.venv`에 정상 설치되는가?
- CUDA 사용 가능 여부가 `True`로 잡히는가?
- 실제 GPU 이름이 A40으로 보이는가?

---

## 이론 — Qwen3-VL 아키텍처 심층 설명

### 먼저 큰 그림부터 이해하기

이 교시의 핵심은 **Qwen3-VL을 다룰 준비를 끝내는 것**이다.  
즉, 다음 교시의 본격 실험을 위해 아래 세 가지를 연결해서 이해해야 한다.

1. 왜 Colab이 아니라 RunPod A40으로 옮겨야 하는가?
2. Qwen3-VL은 기존 VLM과 비교해 무엇이 달라졌는가?
3. 실제로 어떤 구성 요소가 메모리와 추론 품질에 영향을 주는가?

이 세 가지가 연결되면, 이후 실습에서 단순히 코드를 따라 치는 것이 아니라 **왜 이 환경과 모델 구성이 필요한지**를 설명할 수 있게 된다.

### VLM 발전 경로

```
GPT-3 (텍스트만, 2020)
  ↓ 이미지 인코더 추가
CLIP + GPT (2021): 이미지 임베딩 → 텍스트 생성
  ↓ 멀티모달 End-to-End
BLIP (2022): Image-Text 공동 학습
  ↓ Q-Former 도입 (토큰 압축)
BLIP-2 (2023): 고정 LLM + Q-Former 브릿지
  ↓ 완전 파인튜닝
LLaVA (2023): CLIP + MLP + Vicuna 공동 학습
  ↓ 고해상도 / 최신화
InternVL 2.5, Qwen2-VL, Qwen3-VL (2024~2025)
```

이 흐름을 읽을 때는 "비전 인코더를 붙였다"는 사실보다, **이미지 정보를 LLM이 더 잘 받아들이게 만드는 방법이 계속 진화해 왔다**는 점을 보는 것이 중요하다.  
초기에는 이미지 임베딩을 단순히 연결하는 수준이었다면, 최근 모델은 고해상도 처리, 위치 인코딩, 긴 시퀀스 처리, 추론 모드 제어까지 함께 다룬다.

### 주요 VLM 구조 비교

| 모델 | Visual Encoder | Connector | LLM | 특징 |
|---|---|---|---|---|
| LLaVA-1.5 | CLIP ViT-L/14 | 2-layer MLP | Vicuna-13B | 단순 구조, 빠른 학습 |
| BLIP-2 | CLIP ViT-G/14 | Q-Former | OPT/FlanT5 | 토큰 압축 (32개로 축소) |
| InternVL 2.5 | InternViT-6B | MLP | InternLM2.5 | ViT도 대형화 |
| **Qwen3-VL-8B** | **Qwen ViT** | **MLP+M-RoPE** | **Qwen3-8B** | **M-RoPE 2D, DeepStack** |

즉, Qwen3-VL은 단순히 "Qwen LLM에 이미지가 붙은 모델"이 아니라, 시각 위치 정보와 고해상도 처리 전략까지 반영한 **최신형 멀티모달 설계**라고 이해하면 된다.

### Qwen3-VL의 핵심 혁신

**① M-RoPE (Multimodal Rotary Position Embedding)**

기존 LLM의 1D RoPE를 2D로 확장하여 이미지 패치의 공간적 위치를 명시적으로 인코딩한다.

```
텍스트 토큰:    1D 위치 (순서만)
                pos: [0, 1, 2, 3, 4, ...]

이미지 패치:    2D 위치 (행 × 열)
                pos: [(0,0), (0,1), (0,2),
                      (1,0), (1,1), (1,2), ...]
```

→ 모델이 이미지 내 "어디에 있는 패치인지"를 더 정확히 이해

**② DeepStack (동적 고해상도)**

```
입력 이미지 (고해상도)
  ↓ 적응적 타일 분할 (해상도에 따라 1~12 타일)
각 타일 → ViT 처리 → 패치 토큰
  + 전체 썸네일 → ViT 처리 → 패치 토큰
  ↓ 모든 토큰 concat
LLM에 긴 시퀀스로 입력
```

→ 저해상도 전체 맥락 + 고해상도 세부 정보 동시 포착

**③ Instruct vs Thinking 모드**

```python
# Instruct 모드: 즉각적인 답변 생성
question + " /no_think"

# Thinking 모드: 추론 과정을 먼저 생성한 뒤 답변
question + " /think"
# 출력: <think>단계적 추론...</think> 최종 답변
```

여기서 세 혁신을 연결해서 이해하면 더 쉽다.

1. M-RoPE는 **어디에 있는가**를 더 잘 표현한다.
2. DeepStack은 **얼마나 자세히 볼 것인가**를 더 잘 처리한다.
3. Thinking 모드는 **어떻게 답을 구성할 것인가**를 더 잘 드러낸다.

즉, 위치 표현, 해상도 처리, 추론 표현의 세 축이 동시에 강화된 셈이다.

### 입력에서 출력까지 한 번에 보기

Qwen3-VL의 흐름을 간단히 정리하면 아래와 같다.

```
이미지
  ↓ 동적 타일 분할 + ViT 처리
이미지 패치 토큰
  ↓ M-RoPE / Projection
LLM이 읽을 수 있는 시각 토큰

텍스트 프롬프트
  ↓ Tokenizer
텍스트 토큰

시각 토큰 + 텍스트 토큰 통합
  ↓ Qwen3-8B Decoder
최종 답변 생성
```

즉, Qwen3-VL은 "이미지 인코더 + 연결부 + LLM"의 전형적 구조를 가지면서도, 그 내부 설계를 더 정교하게 만든 모델이다.

---

## 모델 로딩 — Notebook

### Step 5. Notebook 생성 및 커널 선택

1. JupyterLab 왼쪽 파일 탐색기에서 `/workspace/AE.2.1` 폴더로 이동한다.
2. 해당 폴더에서 새 Notebook을 생성한다.
3. 커널 선택 창이 뜨면 `Python (.venv) AE.2.1`을 선택한다.
4. 이후 실습 코드는 모두 이 Notebook에서 실행한다.

이 단계에서 확인할 점:

- Notebook 파일이 `/workspace/AE.2.1` 안에 생성되었는가?
- 선택한 커널이 시스템 Python이 아니라 `.venv` 커널인가?

### Step 6. 모델 다운로드 — Terminal

모델 폴더가 비어 있다면 먼저 아래 명령으로 로컬 경로에 모델을 받아 둔다.

```bash
cd /workspace/AE.2.1
source .venv/bin/activate

python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='Qwen/Qwen3-VL-8B-Instruct', local_dir='/workspace/AE.2.1/models/Qwen3-VL-8B-Instruct')"
```

이 단계에서 확인할 점:

- `/workspace/AE.2.1/models/Qwen3-VL-8B-Instruct` 폴더가 실제로 생성되었는가?
- `config.json`, `processor_config.json`, `model-*.safetensors` 같은 파일이 내려왔는가?

`torchvision`을 새로 설치했다면, Notebook 커널을 한 번 재시작한 뒤 다음 셀을 실행한다.

### Step 7. Notebook에서 모델 로딩

In [1]:
from pathlib import Path

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_PATH = Path("/workspace/AE.2.1/models/Qwen3-VL-8B-Instruct")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "모델 폴더가 없습니다. Terminal에서 snapshot_download로 먼저 내려받으세요: "
        "/workspace/AE.2.1/models/Qwen3-VL-8B-Instruct"
    )

print("모델 로딩 중... (약 30초~1분 소요)")

processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
    local_files_only=True,
)
model.eval()
print("✅ 모델 로드 완료")

# 전체 파라미터 수 확인
total = sum(p.numel() for p in model.parameters())
print(f"\n전체 파라미터: {total/1e9:.2f}B")

# VRAM 사용량 확인
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1024**3
    total_mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} VRAM: {used:.1f} / {total_mem:.0f} GB 사용 중")

/workspace/AE.2.1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델 로딩 중... (약 30초~1분 소요)


Loading weights: 100%|██████████| 750/750 [00:02<00:00, 321.03it/s]


✅ 모델 로드 완료

전체 파라미터: 8.77B
GPU 0 VRAM: 16.3 / 44 GB 사용 중


**기대 출력**
```
✅ 모델 로드 완료
전체 파라미터: 8.29B
GPU 0 VRAM: 18.3 / 48 GB 사용 중
```

> A40 48GB 기준 약 18~20GB 사용 → 여유 공간 28GB 이상으로 실습 안정적

이 단계에서 확인할 점:

- `HFValidationError`가 아니라 로컬 경로 기준으로 모델을 읽는가?
- `Torchvision library was not found` 오류 없이 processor가 초기화되는가?
- 모델이 오류 없이 로드되는가?
- BF16 + Flash Attention 조합이 실제로 동작하는가?
- VRAM 사용량이 이후 실습을 진행할 수 있는 수준으로 남아 있는가?

---

## 핵심 개념 정리

```
Qwen3-VL-8B = Qwen ViT + MLP Connector + Qwen3-8B LLM

주요 혁신:
  M-RoPE: 이미지 패치의 2D 위치 인코딩
  DeepStack: 동적 고해상도 (적응적 타일 분할)
  Thinking 모드: 추론 과정 명시적 생성
```

### 혼자 점검하기

아래 질문에 막힘 없이 답하면 이론 이해가 잘 된 것이다.

1. 왜 Qwen3-VL 실습은 Colab보다 RunPod A40 환경이 더 적합한가?
2. Qwen3-VL에서 M-RoPE는 어떤 문제를 해결하려는가?
3. DeepStack은 왜 고해상도 이미지 처리에 유리한가?
4. Instruct 모드와 Thinking 모드는 어떤 상황에서 다르게 쓸 수 있는가?
5. 모델 로딩 뒤 VRAM 사용량을 확인해야 하는 이유는 무엇인가?

---

## 다음 교시 예고

**12교시 (3시간)**: RunPod A40에서 Qwen3-VL 구조를 직접 탐색하고, 멀티모달 추론 실험(해상도 비교, Instruct vs Thinking, 한국어 VQA)을 수행한 뒤 전체 과정을 정리한다.

---

## 부록 | 관찰 과제·혼자 점검 모범 답안

이 부록은 교육생이 환경 전환과 모델 준비 과정을 마친 뒤, 단순 성공 여부를 넘어서 **왜 이런 준비가 필요한지**를 설명할 수 있도록 돕는 해설이다.

### 관찰 과제 모범 답안

**1. A40 GPU가 실제로 연결되어 있는지, 모델 디렉터리가 정상인지 확인**

`nvidia-smi`에서 A40이 보이고, `/workspace/AE.2.1/models/` 아래에 `Qwen3-VL-8B-Instruct/` 폴더가 실제로 존재하면 이후 실습을 진행할 기본 조건이 충족된 것이다. 반대로 `models/`는 있는데 모델 하위 폴더가 비어 있으면, 실행 위치 문제가 아니라 아직 모델이 준비되지 않은 상태일 가능성이 크다. 이 확인이 중요한 이유는, 환경 문제가 생기면 이후 Notebook 오류를 모델 문제로 오해하기 쉽기 때문이다. 즉, 먼저 GPU와 파일 경로를 확인해야 원인 분리가 가능하다.

**2. `uv` 가상환경과 커널 등록 확인**

`.venv`가 실제로 활성화되어 있고, JupyterLab에서 `Python (.venv) AE.2.1` 커널을 선택할 수 있어야 Notebook과 Terminal이 같은 환경을 공유할 수 있다. 이 단계가 빠지면 Terminal에서는 설치가 끝났는데 Notebook에서는 모듈을 찾지 못하는 문제가 자주 생긴다.

**3. 설치 후 CUDA 사용 가능 여부와 GPU 이름 확인**

가상환경을 쓰는 경우에는 전역 Python에 PyTorch가 깔려 있어도 `.venv` 안에서는 별도로 `torch`와 `torchvision`을 설치해야 한다. 특히 Qwen3-VL processor는 내부적으로 `torchvision`을 요구하므로, 이 패키지가 없으면 모델 파일이 있어도 processor 초기화 단계에서 바로 실패한다. 그 뒤에도 `torch.cuda.is_available()`가 `False`면 GPU 가속을 쓰지 못한다. 또 GPU 이름이 예상과 다르면 제공된 인스턴스나 연결 세션이 잘못됐을 가능성도 있다. 따라서 설치 성공만 보는 것이 아니라, 실제 GPU 연산 환경이 준비됐는지까지 확인해야 한다.

**4. 모델 로딩 후 VRAM 사용량 관찰**

모델이 로딩되었다는 사실만으로 충분하지 않다. 실제로 얼마만큼의 VRAM을 쓰는지 확인해야 이후 해상도 증가, Thinking 모드, 배치 확장 같은 실험이 가능한지 판단할 수 있다. 즉, VRAM 확인은 단순 정보 조회가 아니라 이후 실험 가능 범위를 예측하는 단계다.

### 혼자 점검하기 모범 답안

**1. 왜 Qwen3-VL 실습은 Colab보다 RunPod A40 환경이 더 적합한가?**

Qwen3-VL-8B는 모델 크기와 멀티모달 입력 처리 때문에 더 큰 VRAM과 안정적인 GPU 환경이 필요하다. Colab 기본 환경에서는 메모리와 세션 안정성이 부족할 수 있지만, RunPod A40은 48GB VRAM을 제공해 실습을 더 안정적으로 수행할 수 있다.

**2. Qwen3-VL에서 M-RoPE는 어떤 문제를 해결하려는가?**

기존 1D 위치 인코딩만으로는 이미지 패치의 2차원 위치 관계를 충분히 표현하기 어렵다. M-RoPE는 이를 2D로 확장해, 패치가 이미지 안의 어느 행과 열에 있는지를 더 잘 반영하려는 설계다.

**3. DeepStack은 왜 고해상도 이미지 처리에 유리한가?**

고해상도 이미지를 통째로 한 번에 처리하면 토큰 수가 급증해 비용이 커진다. DeepStack은 타일 단위로 세부 정보를 보면서도 전체 썸네일 맥락을 함께 유지해, 전역 정보와 국소 정보를 동시에 다루기 유리하다.

**4. Instruct 모드와 Thinking 모드는 어떤 상황에서 다르게 쓸 수 있는가?**

Instruct 모드는 빠르고 간결한 응답이 필요할 때 적합하다. Thinking 모드는 더 복잡한 추론이나 근거 설명이 필요한 질문에서 유리할 수 있다. 다만 Thinking 모드는 토큰과 시간이 더 많이 들 수 있다.

**5. 모델 로딩 뒤 VRAM 사용량을 확인해야 하는 이유는 무엇인가?**

현재 메모리 사용량을 알아야 이후 실습이 안전하게 가능한지 판단할 수 있기 때문이다. 특히 고해상도 입력, 긴 출력, Thinking 모드, 여러 실험을 이어서 할 때 VRAM 부족으로 중간에 실패할 수 있으므로, 사전에 여유를 확인하는 것이 중요하다.

### 활용 팁

1. 환경 전환 교시에서는 코드보다 먼저 GPU, 경로, 설치 상태를 점검한다.
2. 오류가 나면 모델 코드보다 환경 확인 결과부터 다시 본다.
3. VRAM 수치는 이후 실험 난이도를 조절하는 기준으로 활용한다.
